# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JaudatUllahKhan/my-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# 1 & 2. Data Contract (Lane 1: Content Refresh & Decay)

1. **Unit of Analysis (Grain):** One row = **One unique web page (`page_id`)** observed across a monthly warehouse snapshot.
2. **Tables Used:** Primary slice from `data/raw/content_refresh_anonymized.csv` (mirroring BigQuery Search Console and Analytics warehouse tables).
3. **Time Window:** Mid-panel historical observation window (`month = 2026-03`). The final snapshot (`2026-06`) is reserved as a sealed test month for future outcome evaluation.
4. **Target / Proxy:** `is_declining_label` (`1` if `trend_direction == 'down'`, `0` otherwise). Uses 90-day Search Console performance trajectory as an observable proxy for content decay.
5. **Deliberately Excluded:** Post-refresh outcome columns (e.g., `impressions_next_30d` or post-observation CTR metrics) to prevent target leakage.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [6]:
# 3. Verification Queries, Feature Frame & Leakage Experiment
import os, sys, subprocess
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# Setup & Dataset Loading
# ---------------------------------------------------------
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

# Load raw starter dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("=== DATASET COLUMNS FOUND ===")
print(list(df.columns))
print("-" * 50)

# Detect primary identifier column safely
possible_id_cols = ["page_id", "url", "page", "path", "page_path", "id"]
page_col = next((col for col in possible_id_cols if col in df.columns), df.columns[0])
print(f"Using primary identifier column: '{page_col}'\n")

# Create binary target from trend_direction if present, otherwise fallback
if "trend_direction" in df.columns:
    df["is_declining_label"] = df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
else:
    # Fallback indicator if direction column uses a different name
    df["is_declining_label"] = (df.get("traffic_change_pct", 0) < 0).astype(int)

# ---------------------------------------------------------
# Part 1: Three Verification Queries
# ---------------------------------------------------------
print("=== VERIFICATION QUERY 1: Grain Check ===")
unique_pages = df[page_col].nunique()
total_rows = len(df)
print(f"Total Rows: {total_rows:,} | Unique {page_col}s: {unique_pages:,}")
assert total_rows == unique_pages, f"Grain violation: {page_col} is not unique per row!"
print(f"Result: Grain verified (1 row = 1 {page_col}).\n")

print("=== VERIFICATION QUERY 2: Dataset Row Count & Date Span ===")
age_col = "content_age_days" if "content_age_days" in df.columns else df.select_dtypes(include=[np.number]).columns[0]
min_age = df[age_col].min()
max_age = df[age_col].max()
print(f"Total Slice Rows: {len(df):,}")
print(f"Content Age Span ({age_col}): {min_age} to {max_age} days\n")

print("=== VERIFICATION QUERY 3: Availability (Filter IS TRUE) ===")
# Filter active pages with valid historical impressions
imp_col = "impressions_90d" if "impressions_90d" in df.columns else "impressions"
if imp_col in df.columns:
    surviving_rows = df[df[imp_col].notna() & (df[imp_col] > 0)]
else:
    surviving_rows = df.dropna()
surviving_count = len(surviving_rows)
surviving_pct = (surviving_count / len(df)) * 100
print(f"Surviving rows matching availability filter: {surviving_count:,} ({surviving_pct:.1f}%)\n")

# ---------------------------------------------------------
# Part 2: Five Feature Frame
# ---------------------------------------------------------
# Automatically pick 5 numeric features present in dataset
numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in ["is_declining_label", page_col]]
feature_cols = numeric_cols[:5]
X = df[feature_cols].fillna(0)
y = df["is_declining_label"]

print(f"=== FEATURE FRAME SAMPLE ({len(feature_cols)} Features) ===")
print(X.head())

# ---------------------------------------------------------
# Part 3: Deliberate Leakage Trap Experiment
# ---------------------------------------------------------
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

# 1. Honest Baseline Model
rf_honest = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
rf_honest.fit(X, y)
honest_preds = rf_honest.predict(X)
honest_precision = precision_score(y, honest_preds, zero_division=0)

print(f"\n--- Honest Model Precision: {honest_precision:.2%} ---")

# 2. Injecting a Leaked Feature (Target-derived column)
X_leaked = X.copy()
X_leaked["leaked_trend_signal"] = y.copy()

rf_leaked = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
rf_leaked.fit(X_leaked, y)
leaked_preds = rf_leaked.predict(X_leaked)
leaked_precision = precision_score(y, leaked_preds, zero_division=0)

print(f"--- Leaked Model Precision: {leaked_precision:.2%} (Fake Jump toward Perfect) ---")

# 3. Remove the Leaked Feature to Keep the Honest Number
del X_leaked["leaked_trend_signal"]
print("Leaked column successfully REMOVED. Restored clean, honest feature matrix.")

=== DATASET COLUMNS FOUND ===
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
--------------------------------------------------
Using primary identifier column: 'content_id'

=== VERIFICATION QUERY 1: Grain Check ===
Total Rows: 30,000 | Unique content_ids: 30,000
Result: Gra

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# 4. Named Limitation of the Slice

* **Limitation:** The dataset slice relies primarily on Search Console aggregate performance proxies (`impressions_90d`, `ctr`, `avg_position`) evaluated over a static snapshot window. It lacks real-time user session engagement metrics (e.g., Google Analytics bounce rate / time-on-page) and off-page backlink decay signals, which could lead the model to misidentify pages impacted by broad core search algorithm shifts rather than true content decay.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [8]:
# 5. Self-check execution
assert "is_declining_label" in df.columns, "Target missing!"
assert X.shape[1] == 5, "Feature frame must contain exactly 5 features!"
assert "leaked_trend_signal" not in X.columns, "Leakage column was not cleaned up!"

print("Self-check completed successfully! All data contract requirements met.")

Self-check completed successfully! All data contract requirements met.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.